## Model accuracy
Import libraries

In [ ]:
import os
import json
import utils
import numpy as np
import pandas as pd
import tensorflow as tf

Define inputs

In [ ]:
model_name = "meno_nocorr"

In [ ]:
# Models
models = {
  "meno": "./../03.merged/run01.siso/saving/plasma_cr/SavedModel/",
  "meno_nocorr": "./../02.states/run01.nocorr/saving/plasma_cr_states/SavedModel/"
}
# Paths
path_to_datagen = "./../00.datagen.r02/"
paths = {
  "data": path_to_datagen + "/data/test/",
  "model": models[model_name],
  "saving": f"./output/error/{model_name}/"
}
os.makedirs(paths["saving"], exist_ok=True)
# Error
# > Epsilon value
eps = 1e-10
# > Confidence interval
alpha = 0.95
uncert_name = f"ci_{int(100*alpha)}" if (alpha > 0) else "std"

In [ ]:
# Time grid
tlim = [1e-8, 1e-6]
tdim = 100
t = np.geomspace(*tlim, num=tdim)

In [ ]:
# State vector
variables = ["e$^-$"] \
          + [f"Ar({i+1})" for i in range(31)] \
          + [f"Ar$^+$({i+1})" for i in range(2)] \
          + [f"$T_{i}$" for i in ("h", "e")]

Define data indices

In [ ]:
# Data indices
with open(path_to_datagen + "/data/indices.json") as file:
  _indices = json.load(file)
if (model_name == "meno"):
  indices = {
    "x": [
      _indices["rho_0"],
      _indices["Th_0"],
      _indices["Te_0"],
      _indices["t"]
    ],
    "y": [
      _indices["w"],
      _indices["Th"],
      _indices["Te"]
    ]
  }
elif (model_name == "meno_nocorr"):
  indices = {
    "x": [
      _indices["rho_0"],
      _indices["Th_0"],
      _indices["Te_0"],
      _indices["t"],
      [_indices[k][0] for k in ("n_em", "n_Ar1", "Th", "Te")]
    ],
    "y": [
      _indices["w"]
    ]
  }
else:
  raise ValueError()

Load data and model

In [ ]:
data = utils.load_test_cases(paths["data"])
model = tf.keras.models.load_model(paths["model"])

Set variables names

In [ ]:
names = np.array(list(data.values())[0].columns)
names = np.concatenate([names[i] for i in indices["y"]]).tolist()
names_latex = dict(zip(names, variables))

Compute error

In [ ]:
error = {"l2": [], "mape": []}
for idata in data.values():
  _, y_true, y_pred = utils.predict(model, idata.to_numpy(), indices, t, t_index=3)
  error["l2"].append(utils.error_l2(y_true, y_pred, eps=eps))
  error["mape"].append(utils.error_mape(y_true, y_pred, eps=eps))

Plot and save error

In [ ]:
xticks = [1] + list(range(7,30,7)) + [34]
xticklabels = [variables[i-1] for i in xticks]

In [ ]:
for (k, err) in error.items():
  kerr = np.vstack(err)
  mean, uncert = utils.compute_stats(kerr, axis=0, alpha=alpha)
  kerr = np.vstack([mean, uncert])
  # Plotting error on species
  if (model_name != "meno_nocorr"):
    # > Remove temperatures
    mean = mean[:-2]
    uncert = uncert[:-2]
  utils.plot_err_discrete(
    x=np.arange(len(mean))+1,
    mean=mean,
    uncert=uncert,
    alpha=alpha,
    xlim=[0.5,34.5],
    xticks=xticks,
    xticklabels=xticklabels,
    labels=["Species", r"Error [\%]"],
    scales=["linear", "linear"],
    legend_loc="best",
    figname=paths["saving"]+f"/err_dist_{k}.pdf",
    save=True,
    show=False
  )
  # Saving
  kerr = pd.DataFrame(kerr, columns=names, index=["mean", uncert_name])
  kerr = kerr.sort_values(by=['mean'], axis=1)
  kerr.to_csv(paths["saving"]+f"/{model_name}_{k}.csv", float_format="%.3f")
  utils.write_latex(
    error=kerr,
    filename=paths["saving"]+f"/{model_name}_{k}_latex.txt",
    names=names_latex,
    names_stats=["mean", uncert_name]
  )